In [1]:
# ------------------------------------------------------------
# STEP 1: LOAD SELECTED AGNTWORKS DATA + PREVIEW
# ------------------------------------------------------------

import pandas as pd
import gcsfs
import zipfile
import numpy as np

url = "gs://agntworks-data-dev/wheelsup/raw/AgntWorks.zip"
fs = gcsfs.GCSFileSystem()

with fs.open(url.replace('gs://', '')) as f:
    zf = zipfile.ZipFile(f)
    
    # Load the 4 requested files
    df_bookings = pd.read_csv(zf.open('AgntWorks/booked_flights.csv'), low_memory=False)
    df_quotes = pd.read_csv(zf.open('AgntWorks/requests_for_quotes.csv'), low_memory=False)
    df_search_activity = pd.read_csv(zf.open('AgntWorks/search_activity.csv'), low_memory=False)
    df_summaries = zf.read('AgntWorks/file_summaries.txt').decode('utf-8')

print("✅ All files loaded successfully!")
print("\n" + "="*80)

# Show first 5 rows + basic info for each
print("\n📊 1. df_bookings")
print(f"Shape: {df_bookings.shape}")
print(df_bookings.head())

print("\n" + "="*80)
print("\n📊 2. df_quotes") 
print(f"Shape: {df_quotes.shape}")
print(df_quotes.head())

print("\n" + "="*80)
print("\n📊 3. df_search_activity")
print(f"Shape: {df_search_activity.shape}")
print(df_search_activity.head())

print("\n" + "="*80)
print("\n📄 4. df_summaries (first 2000 chars)")
print(df_summaries[:2000])

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)


✅ All files loaded successfully!


📊 1. df_bookings
Shape: (258370, 43)
   flightId  atlasreservationid flightStatus  legOrder  flightoriginAirportId  \
0   1208327              794518    Cancelled         1                    810   
1   1208330              794521        Flown         1                    333   
2   1208333              794523        Flown         1                   1047   
3   1208341              794528    Cancelled         2                    391   
4   1208344              794530        Flown         2                   2983   

  flightoriginAirport     flightoriginAirportName flightoriginAirportCity  \
0                KTEB                   Teterboro               Teterboro   
1                KPBI    Palm Beach International         West Palm Beach   
2                KBKL             Burke Lakefront               Cleveland   
3                KYNG  Youngstown Warren Regional              Youngstown   
4                KHXD                 Hilton Head      H

In [2]:
# ------------------------------------------------------------
# CHECK NULLS IN df_bookings
# ------------------------------------------------------------

print("📊 df_bookings NULL ANALYSIS")
print(f"Total rows: {len(df_bookings):,}")
print("\n" + "="*60)

# Total nulls per column
null_counts = df_bookings.isnull().sum()
null_pct = (null_counts / len(df_bookings)) * 100

null_summary = pd.DataFrame({
    'Null_Count': null_counts,
    'Null_%': null_pct.round(2)
}).sort_values('Null_Count', ascending=False)

print(null_summary[null_summary['Null_Count'] > 0].head(10))
print(f"\nColumns with NO nulls: {(null_summary['Null_Count'] == 0).sum()}/{len(null_summary)}")

# Quick summary stats
print(f"\n💡 Quick stats:")
print(f"  - Columns with >50% nulls: {(null_pct > 50).sum()}")
print(f"  - Columns with 0% nulls: {(null_pct == 0).sum()}")
print(f"  - Total nulls across dataset: {null_counts.sum():,}")

📊 df_bookings NULL ANALYSIS
Total rows: 258,370

                                  Null_Count  Null_%
flightActualArrivalTime                74454   28.82
flightActualDepartureTime              72479   28.05
flightrequestedAircraftCabinName       66474   25.73
flightrequestedAircraftCabinId         66474   25.73
flighttailNumber                       22857    8.85
flightactualAircraftId                 22700    8.79
flightactualAircraftTypeName           22700    8.79
flightActualFlightHours                18343    7.10
flightActualBilledHours                12048    4.66
bookerContactId                         7792    3.02

Columns with NO nulls: 22/43

💡 Quick stats:
  - Columns with >50% nulls: 0
  - Columns with 0% nulls: 22
  - Total nulls across dataset: 403,048


In [3]:
# ------------------------------------------------------------
# CHECK UNIQUE VALUES: reservationStatus
# ------------------------------------------------------------

print("📊 df_bookings['reservationStatus'] UNIQUE VALUES")
print("="*50)

status_counts = df_bookings['reservationStatus'].value_counts()
status_pct = df_bookings['reservationStatus'].value_counts(normalize=True) * 100

status_summary = pd.DataFrame({
    'Count': status_counts,
    'Percentage': status_pct.round(2)
}).sort_values('Count', ascending=False)

print(status_summary)
print(f"\nTotal unique statuses: {df_bookings['reservationStatus'].nunique()}")

📊 df_bookings['reservationStatus'] UNIQUE VALUES
                    Count  Percentage
reservationStatus                    
Quote              171363       66.32
Pending             85323       33.02
Incomplete            922        0.36
Update Required       482        0.19
Booked                280        0.11

Total unique statuses: 5


In [4]:
# ------------------------------------------------------------
# ANALYZE reservationCreateDate
# ------------------------------------------------------------

print("📊 reservationCreateDate ANALYSIS")
print("="*50)

# Check for nulls
null_date_count = df_bookings['reservationCreateDate'].isnull().sum()
print(f"Null dates: {null_date_count:,} ({null_date_count/len(df_bookings)*100:.2f}%)")

# Convert to datetime if needed and get min/max
df_bookings['reservationCreateDate'] = pd.to_datetime(df_bookings['reservationCreateDate'], errors='coerce')
date_range = df_bookings['reservationCreateDate'].agg(['min', 'max', 'count'])
print(f"\nDate range:")
print(f"  Min: {date_range['min']}")
print(f"  Max: {date_range['max']}")
print(f"  Valid dates: {date_range['count']:,}")

# By reservationStatus
print(f"\n📅 Dates by reservationStatus:")
status_dates = df_bookings.groupby('reservationStatus')['reservationCreateDate'].agg(['min', 'max', 'count']).round(-3)
print(status_dates)

# Days covered
days_span = (date_range['max'] - date_range['min']).days
print(f"\n⏱️  Total time span: {days_span} days (~{days_span/30:.1f} months)")

📊 reservationCreateDate ANALYSIS
Null dates: 0 (0.00%)

Date range:
  Min: 2024-04-01 06:06:30+00:00
  Max: 2026-04-01 22:45:29+00:00
  Valid dates: 258,370

📅 Dates by reservationStatus:
                                        min                       max   count
reservationStatus                                                            
Booked            2024-04-01 11:12:44+00:00 2025-10-15 18:22:12+00:00       0
Incomplete        2024-04-01 07:21:01+00:00 2026-02-15 06:59:10+00:00    1000
Pending           2024-04-01 08:50:14+00:00 2026-04-01 21:14:41+00:00   85000
Quote             2024-04-01 06:06:30+00:00 2026-04-01 22:45:29+00:00  171000
Update Required   2024-04-01 11:41:06+00:00 2025-02-27 10:13:14+00:00       0

⏱️  Total time span: 730 days (~24.3 months)


In [5]:
# ------------------------------------------------------------
# CHECK MULTI-LEG BOOKINGS (atlasreservationid → legOrder)
# ------------------------------------------------------------

print("🔍 MULTI-LEG BOOKING ANALYSIS")
print("="*60)

# Count legs per atlasreservationid
legs_per_booking = df_bookings.groupby('atlasreservationid')['legOrder'].nunique().value_counts().sort_index()
print("Legs per booking:")
print(legs_per_booking)
print(f"\nTotal unique bookings: {len(df_bookings['atlasreservationid'].unique()):,}")

# Multi-leg breakdown
multi_leg_bookings = df_bookings['atlasreservationid'].value_counts()
print(f"\n🏃 Single-leg bookings (1 leg): {len(multi_leg_bookings[multi_leg_bookings == 1])} ({len(multi_leg_bookings[multi_leg_bookings == 1])/len(multi_leg_bookings)*100:.1f}%)")
print(f"🔄 Multi-leg bookings (2+ legs): {len(multi_leg_bookings[multi_leg_bookings > 1])} ({len(multi_leg_bookings[multi_leg_bookings > 1])/len(multi_leg_bookings)*100:.1f}%)")

# Show sample multi-leg booking
multi_leg_sample = multi_leg_bookings[multi_leg_bookings > 1].index[0]
print(f"\n📋 Sample multi-leg booking (ID: {multi_leg_sample}):")
print(df_bookings[df_bookings['atlasreservationid'] == multi_leg_sample][['atlasreservationid', 'legOrder', 'flightStatus', 'flightoriginAirport', 'flightcost']].head())

# legOrder distribution
print(f"\n📊 legOrder values found: {sorted(df_bookings['legOrder'].unique())}")
print("legOrder counts:")
print(df_bookings['legOrder'].value_counts().sort_index())

🔍 MULTI-LEG BOOKING ANALYSIS
Legs per booking:
legOrder
1     27292
2     23315
3      1447
4       637
5        94
6        43
7         4
8        11
9         1
10        1
Name: count, dtype: int64

Total unique bookings: 52,845

🏃 Single-leg bookings (1 leg): 6984 (13.2%)
🔄 Multi-leg bookings (2+ legs): 45861 (86.8%)

📋 Sample multi-leg booking (ID: 819319):
       atlasreservationid  legOrder flightStatus flightoriginAirport  \
21577              819319         8    Cancelled                KMYR   
21578              819319        10    Cancelled                KMYR   
23256              819319         8    Cancelled                KMYR   
23257              819319        10    Cancelled                KMYR   
24408              819319         8    Cancelled                KMYR   

       flightcost  
21577         0.0  
21578         0.0  
23256         0.0  
23257         0.0  
24408         0.0  

📊 legOrder values found: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np

In [6]:
# ------------------------------------------------------------
# UNIQUE AIRCRAFT TYPES: flightactualAircraftTypeName
# ------------------------------------------------------------

print("✈️  UNIQUE AIRCRAFT TYPES")
print("="*50)

# Get unique aircraft names (excluding nulls)
unique_aircraft = df_bookings['flightactualAircraftTypeName'].dropna().unique()
aircraft_counts = df_bookings['flightactualAircraftTypeName'].value_counts()

print(f"Total unique aircraft types: {len(unique_aircraft)}")
print(f"\nTop 10 most common aircraft:")
print(aircraft_counts.head(10))

print(f"\nAll unique aircraft types:")
for aircraft in sorted(unique_aircraft):
    count = aircraft_counts.get(aircraft, 0)
    print(f"  {aircraft} ({count:,} flights)")

✈️  UNIQUE AIRCRAFT TYPES
Total unique aircraft types: 117

Top 10 most common aircraft:
flightactualAircraftTypeName
King Air 350i    59061
CE-750           43066
Hawker 400XP     30710
Phenom 300       28606
CE-560-XL        27194
CE-525B          11829
CL-300            6044
BE-400-A          3374
Hawker 800XP      2996
LR-60             2087
Name: count, dtype: int64

All unique aircraft types:
  400XT (167 flights)
  BD-3500 (18 flights)
  BD-700-1A10 (10 flights)
  BD-700-1A11 (60 flights)
  BE-400 (195 flights)
  BE-400-A (3,374 flights)
  BHT- 430 (12 flights)
  CE-208B (7 flights)
  CE-500 (673 flights)
  CE-525 (71 flights)
  CE-525+ (6 flights)
  CE-525, CJ1 (25 flights)
  CE-525, CJ1+ (122 flights)
  CE-525A, CJ2 (189 flights)
  CE-525A, CJ2+ (80 flights)
  CE-525B (11,829 flights)
  CE-525B+ (50 flights)
  CE-525C (259 flights)
  CE-550 (90 flights)
  CE-550 Bravo (1,846 flights)
  CE-560 (763 flights)
  CE-560 Encore (420 flights)
  CE-560 XLS (344 flights)
  CE-560 XLS+ 

In [7]:
# ------------------------------------------------------------
# flightStatus BY NULL COLUMNS ANALYSIS
# ------------------------------------------------------------

print("🔍 flightStatus WHERE COLUMNS HAVE NULLs")
print("="*70)

# Top null columns from earlier
top_null_cols = ['flightActualArrivalTime', 'flightActualDepartureTime', 
                'flightrequestedAircraftCabinName', 'flightrequestedAircraftCabinId',
                'flighttailNumber', 'flightactualAircraftId']

for col in top_null_cols:
    if col in df_bookings.columns:
        null_mask = df_bookings[col].isnull()
        status_with_nulls = df_bookings.loc[null_mask, 'flightStatus'].value_counts()
        
        print(f"\n📊 {col} (Nulls: {null_mask.sum():,})")
        print(status_with_nulls)
        print(f"  → {len(status_with_nulls)} unique statuses")

# Summary table
print("\n" + "="*70)
print("SUMMARY: Nulls by flightStatus")
null_by_status = df_bookings.groupby('flightStatus')[top_null_cols].sum()
print(null_by_status)

🔍 flightStatus WHERE COLUMNS HAVE NULLs

📊 flightActualArrivalTime (Nulls: 74,454)
flightStatus
Flown             37210
Cancelled         33476
Quote              2830
Booked              508
Pending             394
Cancel Pending       19
Incomplete            9
Logged                6
Approved              2
Name: count, dtype: int64
  → 9 unique statuses

📊 flightActualDepartureTime (Nulls: 72,479)
flightStatus
Flown             35272
Cancelled         33449
Quote              2822
Booked              506
Pending             394
Cancel Pending       19
Incomplete            9
Logged                6
Approved              2
Name: count, dtype: int64
  → 9 unique statuses

📊 flightrequestedAircraftCabinName (Nulls: 66,474)
flightStatus
Flown             57897
Cancelled          8156
Pending             358
Booked               53
Quote                 4
Cancel Pending        4
Approved              2
Name: count, dtype: int64
  → 7 unique statuses

📊 flightrequestedAircraftCabinId (Nu

IOStream.flush timed out


                                          flightActualArrivalTime  \
flightStatus                                                        
Approved                                                        0   
Booked                                                          0   
Cancel Pending                                                  0   
Cancelled       2025-01-04T12:20:00.000Z2025-01-26T19:33:00.00...   
Flown           2024-04-04T13:35:00.000Z2024-04-06T11:44:00.00...   
Incomplete                                                      0   
Logged                                                          0   
Pending         2024-04-05T15:39:00.000Z2024-04-05T15:39:00.00...   
Quote                                                           0   

                                        flightActualDepartureTime  \
flightStatus                                                        
Approved                                                        0   
Booked           2026-04-23T15:20

In [8]:
# ------------------------------------------------------------
# UNIQUE flightStatus VALUES (ALL)
# ------------------------------------------------------------

print("📊 ALL UNIQUE flightStatus VALUES")
print("="*40)

unique_statuses = df_bookings['flightStatus'].unique()
status_counts = df_bookings['flightStatus'].value_counts()

print(f"Total unique flightStatus: {len(unique_statuses)}")
print("\nFull breakdown:")
for status in sorted(unique_statuses):
    count = status_counts.get(status, 0)
    pct = (count / len(df_bookings)) * 100
    print(f"  {status:<20} | {count:>8,} | {pct:>5.1f}%")

print(f"\nTotal rows: {len(df_bookings):,}")

📊 ALL UNIQUE flightStatus VALUES
Total unique flightStatus: 9

Full breakdown:
  Approved             |        2 |   0.0%
  Booked               |      508 |   0.2%
  Cancel Pending       |       19 |   0.0%
  Cancelled            |   33,572 |  13.0%
  Flown                |  220,957 |  85.5%
  Incomplete           |        9 |   0.0%
  Logged               |        6 |   0.0%
  Pending              |      467 |   0.2%
  Quote                |    2,830 |   1.1%

Total rows: 258,370


In [9]:
# ------------------------------------------------------------
# UNIQUE operatorName VALUES
# ------------------------------------------------------------

print("🏢 UNIQUE OPERATOR NAMES")
print("="*50)

unique_operators = df_bookings['operatorName'].dropna().unique()
operator_counts = df_bookings['operatorName'].value_counts()

print(f"Total unique operators: {len(unique_operators)}")
print(f"\nTop 10 operators by flight volume:")
print(operator_counts.head(10))

print(f"\nAll unique operator names:")
for operator in sorted(unique_operators):
    count = operator_counts.get(operator, 0)
    print(f"  {operator:<30} ({count:,} flights)")

🏢 UNIQUE OPERATOR NAMES
Total unique operators: 184

Top 10 operators by flight volume:
operatorName
Wheels Up Private Jets, LLC    159448
Wheels Up Partners, LLC         19868
Pending Operator                17392
Baker Aviation                   6871
Premier Private Jets             6224
GrandView Aviation               6042
Ventura Air Services             4925
Jet Linx Aviation                2923
Circadian Aviation LLC           2392
Pending Recovery                 2166
Name: count, dtype: int64

All unique operator names:
  AB Jets                        (1,066 flights)
  ATI Jet                        (765 flights)
  Advanced Air, LLC              (15 flights)
  Aero Air Charter               (139 flights)
  Aero-Tech Services             (6 flights)
  Air Associates Charter         (4 flights)
  Air Center Helicopters Inc     (3 flights)
  AirCharters Worldwide          (4 flights)
  AirMed International           (3 flights)
  AirStat                        (49 flights)
  Air

In [10]:
# ------------------------------------------------------------
# NULL VALUES WHERE flightStatus = 'Flown' 
# ------------------------------------------------------------

print("🔍 NULLs in Flown Flights ONLY")
print("="*60)

# Filter to Flown flights only
flown_rows = df_bookings[df_bookings['flightStatus'] == 'Flown']

print(f"Flown flights analyzed: {len(flown_rows):,}")
print()

# Check nulls in Flown flights
flown_nulls = flown_rows.isnull().sum()
flown_null_pct = (flown_nulls / len(flown_rows)) * 100

flown_null_summary = pd.DataFrame({
    'Flown_Null_Count': flown_nulls,
    'Flown_Null_%': flown_null_pct.round(2)
}).query("Flown_Null_Count > 0")

print("Columns with nulls in Flown flights:")
print(flown_null_summary.sort_values('Flown_Null_Count', ascending=False))

print(f"\n✅ Columns with ZERO nulls in Flown flights: {len(flown_nulls[flown_nulls == 0])}/{len(flown_nulls)}")
print(f"💯 Data completeness for Flown flights: {(1 - flown_nulls.mean())*100:.1f}%")

🔍 NULLs in Flown Flights ONLY
Flown flights analyzed: 220,957

Columns with nulls in Flown flights:
                                  Flown_Null_Count  Flown_Null_%
flightrequestedAircraftCabinName             57897         26.20
flightrequestedAircraftCabinId               57897         26.20
flightActualArrivalTime                      37210         16.84
flightActualDepartureTime                    35272         15.96
bookerContactId                               6489          2.94
flightActualBilledHours                       6336          2.87
flightActualFlightHours                       4996          2.26
flightdestinationAirportState                 3927          1.78
flightoriginAirportState                      3835          1.74
flightcost                                     509          0.23
flighttailNumber                               448          0.20
flightactualAircraftId                         441          0.20
flightactualAircraftTypeName                   441     

In [11]:
# ------------------------------------------------------------
# LIST ALL 43 COLUMN NAMES
# ------------------------------------------------------------

print("📋 df_bookings: ALL 43 COLUMNS")
print("="*50)

# Simple column list
print("Column names:")
for i, col in enumerate(df_bookings.columns, 1):
    print(f"{i:2d}. {col}")

print(f"\nTotal columns: {len(df_bookings.columns)}")
print(f"Shape reminder: {df_bookings.shape}")

📋 df_bookings: ALL 43 COLUMNS
Column names:
 1. flightId
 2. atlasreservationid
 3. flightStatus
 4. legOrder
 5. flightoriginAirportId
 6. flightoriginAirport
 7. flightoriginAirportName
 8. flightoriginAirportCity
 9. flightoriginAirportState
10. flightoriginAirportCountry
11. flightdestinationAirportId
12. flightdestinationAirport
13. flightdestinationAirportName
14. flightdestinationAirportCity
15. flightdestinationAirportState
16. flightdestinationAirportCountry
17. flightEstimatedDepartureTime
18. flightEstimatedArrivalTime
19. flightEstimatedFlightHours
20. flightEstimatedBilledHours
21. flightActualDepartureTime
22. flightActualArrivalTime
23. flightActualFlightHours
24. flightActualBilledHours
25. flightrequestedAircraftType
26. flightrequestedAircraftCabinId
27. flightrequestedAircraftCabinName
28. flightrequestedAircraftTypeId
29. flightrequestedAircraftTypeName
30. flighttailNumber
31. flightactualAircraftCabinId
32. flightactualAircraftCabinName
33. flightactualAircraftId


In [12]:
# ------------------------------------------------------------
# COUNT ICAO LEGS PER SEARCH JOURNEY
# ------------------------------------------------------------

print("🔍 SEARCH ACTIVITY: ICAO LEGS PER JOURNEY")
print("="*60)

# Count ICAO airports per route (split by '-')
df_search_activity['icao_count'] = df_search_activity['route'].str.split('-').str.len()

# Breakdown by number of legs
legs_distribution = df_search_activity['icao_count'].value_counts().sort_index()
legs_pct = df_search_activity['icao_count'].value_counts(normalize=True) * 100

print("Journey complexity (ICAO airports per search):")
legs_summary = pd.DataFrame({
    'Searches': legs_distribution,
    'Percentage': legs_pct.round(2)
})
print(legs_summary)

print(f"\n📊 SUMMARY:")
print(f"  Total searches: {len(df_search_activity):,}")
print(f"  Simple (2 airports): {legs_distribution.get(2,0):,} ({legs_pct.get(2,0):.1f}%)")
print(f"  Roundtrips (3 airports): {legs_distribution.get(3,0):,} ({legs_pct.get(3,0):.1f}%)")
print(f"  Multi-city (4+ airports): {legs_distribution[legs_distribution.index >= 4].sum():,} ({legs_pct[legs_pct.index >= 4].sum():.1f}%)")

# Top complex routes
print(f"\n🏆 Top 5 most complex routes (ICAOs):")
print(df_search_activity.nlargest(5, 'icao_count')[['route', 'icao_count', 'isCartAbandoned']])

print(f"\n📈 Cart abandonment by complexity:")
abandon_by_legs = df_search_activity.groupby('icao_count')['isCartAbandoned'].mean()
print(abandon_by_legs.round(3))

🔍 SEARCH ACTIVITY: ICAO LEGS PER JOURNEY
Journey complexity (ICAO airports per search):
            Searches  Percentage
icao_count                      
2             367946       48.05
3             384172       50.17
4               9046        1.18
5               3534        0.46
6                528        0.07
7                304        0.04
8                 94        0.01
9                 54        0.01
10                 4        0.00

📊 SUMMARY:
  Total searches: 765,682
  Simple (2 airports): 367,946 (48.1%)
  Roundtrips (3 airports): 384,172 (50.2%)
  Multi-city (4+ airports): 13,564 (1.8%)

🏆 Top 5 most complex routes (ICAOs):
                                                    route  icao_count  \
258665  KCPS-KSDF-KBHM-KMGM-KMCI-KMSP-KMOB-KBWI-KIAD-KCLT          10   
258667  KCPS-KSDF-KBHM-KMGM-KMCI-KMSP-KMOB-KBWI-KIAD-KCLT          10   
258668  KCPS-KSDF-KBHM-KMGM-KMCI-KMSP-KMOB-KBWI-KIAD-KCLT          10   
633239  KMDW-KAKR-KMIV-KABE-KCMI-KPIA-KMKC-KIAH-KDFW-KFTY

TypeError: Expected numeric dtype, got object instead.

In [ ]:
# ------------------------------------------------------------
# UNIQUE flightStatus VALUES (COMPLETE)
# ------------------------------------------------------------

print("📊 df_bookings['flightStatus'] UNIQUE VALUES")
print("="*50)

# Get all unique values + counts
status_counts = df_bookings['flightStatus'].value_counts()
status_pct = df_bookings['flightStatus'].value_counts(normalize=True) * 100

status_summary = pd.DataFrame({
    'Count': status_counts,
    'Percentage': status_pct.round(2)
}).sort_values('Count', ascending=False)

print(status_summary)
print(f"\nTotal unique flightStatus values: {df_bookings['flightStatus'].nunique()}")
print(f"Total rows: {len(df_bookings):,}")

In [ ]:
# ------------------------------------------------------------
# UNIQUE VALUE COUNTS FOR KEY COLUMNS
# ------------------------------------------------------------

columns_to_check = [
    'flighttailNumber', 'flightactualAircraftCabinId', 'flightactualAircraftCabinName',
    'flightactualAircraftId', 'flightactualAircraftTypeName', 'operatorName', 
    'operatorType', 'flightcost', 'reservationId', 'reservationStatus', 
    'reservationCreateDate', 'bookerMemberId', 'bookerContactId', 'passengerId'
]

print("📊 UNIQUE VALUES PER COLUMN (df_bookings)")
print("="*70)

unique_summary = {}
for col in columns_to_check:
    if col in df_bookings.columns:
        unique_count = df_bookings[col].nunique(dropna=False)
        null_count = df_bookings[col].isnull().sum()
        unique_summary[col] = {'unique': unique_count, 'nulls': null_count}
        
        print(f"{col:<30} | {unique_count:>8,} unique | {null_count:>6,} nulls")

print(f"\nTotal columns analyzed: {len(unique_summary)}")

In [ ]:
# ------------------------------------------------------------
# ACTUAL UNIQUE VALUES FOR LOW-CARDINALITY COLUMNS
# ------------------------------------------------------------

low_card_cols = [
    'flightactualAircraftCabinId',
    'flightactualAircraftCabinName', 
    'operatorType',
    'reservationStatus'
]

print("📋 ACTUAL UNIQUE VALUES (Low Cardinality Columns)")
print("="*70)

for col in low_card_cols:
    print(f"\n{col}:")
    print("-" * 40)
    unique_vals = df_bookings[col].dropna().unique()
    for val in sorted(unique_vals):
        count = df_bookings[df_bookings[col] == val].shape[0]
        print(f"  {val:<25} ({count:>6,} rows)")
    print(f"  [NULL]                   ({df_bookings[col].isnull().sum():>6,} rows)")
    print()

In [ ]:
# ------------------------------------------------------------
# UNIQUE VALUES: flightrequestedAircraftType, legOrder (x2)
# ------------------------------------------------------------

cols_to_check = ['flightrequestedAircraftType', 'legOrder']

print("📋 UNIQUE VALUES - Requested Aircraft & Leg Order")
print("="*60)

for col in cols_to_check:
    print(f"\n{col}:")
    print("-" * 40)
    unique_vals = df_bookings[col].dropna().unique()
    for val in sorted(unique_vals):
        count = df_bookings[df_bookings[col] == val].shape[0]
        print(f"  {val:<25} ({count:>7,} rows)")
    print(f"  [NULL]                   ({df_bookings[col].isnull().sum():>7,} rows)")
    print()

In [ ]:
# ------------------------------------------------------------
# flightStatus UNIQUE VALUES (COMPLETE LIST)
# ------------------------------------------------------------

print("📊 df_bookings['flightStatus'] UNIQUE VALUES")
print("="*50)

# Get all unique values + counts
status_counts = df_bookings['flightStatus'].value_counts()
status_pct = df_bookings['flightStatus'].value_counts(normalize=True) * 100

status_summary = pd.DataFrame({
    'Count': status_counts,
    'Percentage': status_pct.round(2)
}).sort_values('Count', ascending=False)

print(status_summary)
print(f"\nTotal unique flightStatus values: {df_bookings['flightStatus'].nunique()}")
print(f"Total rows: {len(df_bookings):,}")

In [ ]:
# ------------------------------------------------------------
# flightoriginAirportCountry UNIQUE VALUES
# ------------------------------------------------------------

print("🌍 flightoriginAirportCountry UNIQUE VALUES")
print("="*50)

country_counts = df_bookings['flightoriginAirportCountry'].value_counts()
country_pct = df_bookings['flightoriginAirportCountry'].value_counts(normalize=True) * 100

country_summary = pd.DataFrame({
    'Count': country_counts,
    'Percentage': country_pct.round(2)
}).sort_values('Count', ascending=False)

print(country_summary)
print(f"\nTotal unique countries: {df_bookings['flightoriginAirportCountry'].nunique()}")
print(f"Total rows: {len(df_bookings):,}")

In [ ]:
# ------------------------------------------------------------
# flightoriginAirportId & flightdestinationAirportId UNIQUE VALUES
# ------------------------------------------------------------

cols = ['flightoriginAirportId', 'flightdestinationAirportId']

print("🛫 AIRPORT ID UNIQUE VALUES (Top 20 each)")
print("="*70)

for col in cols:
    print(f"\n{col}:")
    print("-" * 50)
    
    # Top 20 + totals
    top_airports = df_bookings[col].value_counts().head(20)
    nulls = df_bookings[col].isnull().sum()
    
    for airport, count in top_airports.items():
        print(f"  {airport:<8} ({count:>8,} rows)")
    
    print(f"  [NULL]     ({nulls:>8,} rows)")
    print(f"  TOTAL UNIQUE: {df_bookings[col].nunique(dropna=False):>5}")
    print()

In [ ]:
# ------------------------------------------------------------
# flightoriginAirport & flightdestinationAirport UNIQUE VALUES
# ------------------------------------------------------------

cols = ['flightoriginAirport', 'flightdestinationAirport']

print("🏢 AIRPORT NAMES UNIQUE VALUES (Top 20 each)")
print("="*70)

for col in cols:
    print(f"\n{col}:")
    print("-" * 50)
    
    # Top 20 + totals
    top_airports = df_bookings[col].value_counts().head(20)
    nulls = df_bookings[col].isnull().sum()
    
    for airport, count in top_airports.items():
        print(f"  {airport:<30} ({count:>8,} rows)")
    
    print(f"\n  [NULL]                   ({nulls:>8,} rows)")
    print(f"  TOTAL UNIQUE: {df_bookings[col].nunique(dropna=False):>5}")
    print()

In [ ]:
# ------------------------------------------------------------
# flightrequestedAircraftTypeId UNIQUE VALUES
# ------------------------------------------------------------

print("🛩️ flightrequestedAircraftTypeId UNIQUE VALUES")
print("="*50)

typeid_counts = df_bookings['flightrequestedAircraftTypeId'].value_counts()
typeid_pct = df_bookings['flightrequestedAircraftTypeId'].value_counts(normalize=True) * 100

typeid_summary = pd.DataFrame({
    'Count': typeid_counts,
    'Percentage': typeid_pct.round(2)
}).sort_values('Count', ascending=False)

print(typeid_summary)
print(f"\nTotal unique Type IDs: {df_bookings['flightrequestedAircraftTypeId'].nunique()}")
print(f"Total rows: {len(df_bookings):,}")
print(f"Nulls: {df_bookings['flightrequestedAircraftTypeId'].isnull().sum()}")

In [ ]:
# METHOD 1: Zero-cost Flown flights (primary signal)
df_reposition = df_bookings[
    (df_bookings['flightStatus'] == 'Flown') & 
    (df_bookings['flightcost'] == 0)
]
print(f"Repositioning flights: {len(df_reposition):,} legs")
print(f"Percentage of Flown: {len(df_reposition)/220957*100:.1f}%")

In [ ]:
# ------------------------------------------------------------
# flightrequestedAircraftType UNIQUE VALUES (Aircraft MODEL NAMES)
# ------------------------------------------------------------

print("✈️ flightrequestedAircraftType UNIQUE VALUES")
print("="*60)

type_counts = df_bookings['flightrequestedAircraftType'].value_counts()
type_pct = df_bookings['flightrequestedAircraftType'].value_counts(normalize=True) * 100

type_summary = pd.DataFrame({
    'Count': type_counts,
    'Percentage': type_pct.round(2)
}).sort_values('Count', ascending=False).head(20)

print(type_summary)
print(f"\nTotal unique aircraft models: {df_bookings['flightrequestedAircraftType'].nunique()}")
print(f"Total rows: {len(df_bookings):,}")
print(f"Nulls: {df_bookings['flightrequestedAircraftType'].isnull().sum()}")

In [ ]:
# ------------------------------------------------------------
# flightrequestedAircraftTypeId UNIQUE VALUES (Top 20)
# ------------------------------------------------------------

print("🔢 flightrequestedAircraftTypeId - TOP 20")
print("="*50)

top_types = df_bookings['flightrequestedAircraftTypeId'].value_counts().head(20)
for type_id, count in top_types.items():
    print(f"ID {type_id:<4} → {count:>6,} legs ({count/258370*100:.1f}%)")

print(f"\nTotal unique: {df_bookings['flightrequestedAircraftTypeId'].nunique()}")
print(f"Top 20 cover: {top_types.sum()/len(df_bookings)*100:.1f}% of demand")

In [ ]:
# ------------------------------------------------------------
# flightactualAircraftId UNIQUE VALUES
# ------------------------------------------------------------

print("🛩️ flightactualAircraftId UNIQUE VALUES")
print("="*50)

actual_id_counts = df_bookings['flightactualAircraftId'].value_counts()
actual_id_pct = df_bookings['flightactualAircraftId'].value_counts(normalize=True) * 100

actual_id_summary = pd.DataFrame({
    'Count': actual_id_counts,
    'Percentage': actual_id_pct.round(2)
}).sort_values('Count', ascending=False).head(20)

print(actual_id_summary)
print(f"\nTotal unique aircraft IDs: {df_bookings['flightactualAircraftId'].nunique()}")
print(f"Total rows: {len(df_bookings):,}")
print(f"Nulls: {df_bookings['flightactualAircraftId'].isnull().sum()}")

In [ ]:
# ------------------------------------------------------------
# flightactualAircraftTypeName UNIQUE VALUES (ACTUAL FLIGHT AIRCRAFT)
# ------------------------------------------------------------

print("✈️ flightactualAircraftTypeName UNIQUE VALUES")
print("="*60)

actual_type_counts = df_bookings['flightactualAircraftTypeName'].value_counts()
actual_type_pct = df_bookings['flightactualAircraftTypeName'].value_counts(normalize=True) * 100

actual_type_summary = pd.DataFrame({
    'Count': actual_type_counts,
    'Percentage': actual_type_pct.round(2)
}).sort_values('Count', ascending=False).head(20)

print(actual_type_summary)
print(f"\nTotal unique actual aircraft types: {df_bookings['flightactualAircraftTypeName'].nunique()}")
print(f"Total rows: {len(df_bookings):,}")
print(f"Nulls: {df_bookings['flightactualAircraftTypeName'].isnull().sum()}")

In [ ]:
# ------------------------------------------------------------
# reservationStatus for flightStatus = 'Flown' ONLY
# ------------------------------------------------------------

print("📋 reservationStatus WHERE flightStatus = 'Flown'")
print("="*60)

flown_data = df_bookings[df_bookings['flightStatus'] == 'Flown']
status_counts = flown_data['reservationStatus'].value_counts()
status_pct = flown_data['reservationStatus'].value_counts(normalize=True) * 100

status_summary = pd.DataFrame({
    'Count': status_counts,
    'Percentage': status_pct.round(2)
}).sort_values('Count', ascending=False)

print(status_summary)
print(f"\nTotal Flown legs analyzed: {len(flown_data):,}")
print(f"Unique reservationStatus in Flown: {flown_data['reservationStatus'].nunique()}")

In [ ]:
# VALIDATE Flown data completeness
flown_data = df_bookings[df_bookings['flightStatus'] == 'Flown']

required_fields = [
    'flightId', 'atlasreservationid', 'legOrder', 'flightoriginAirport', 
    'flightdestinationAirport', 'flightActualDepartureTime', 
    'flightActualArrivalTime', 'flightActualFlightHours', 
    'flightActualBilledHours', 'flightactualAircraftCabinName', 
    'flightactualAircraftTypeName', 'operatorName', 'operatorType'
]

for field in required_fields:
    null_pct = flown_data[field].isnull().mean() * 100
    print(f"{field:<30} | {null_pct:>5.1f}% nulls")

In [ ]:
# 1. Cross-check null times vs flightcost
null_times = df_bookings[
    (df_bookings['flightStatus']=='Flown') & 
    (df_bookings['flightActualDepartureTime'].isnull())
]
print(f"Null times WITH revenue: {len(null_times[null_times['flightcost']>0]):,} legs")

# 2. Check if same aircraft flew multiple legs that day
print("Pattern check for suspicious flights...")

In [ ]:
suspicious = df_bookings[
    (df_bookings['flightStatus']=='Flown') & 
    (df_bookings['flightcost']>0) & 
    (df_bookings['flightActualDepartureTime'].isnull())
]

print("🔍 SUSPICIOUS FLIGHT PATTERNS:")

# 1. AIRCRAFT TYPE (short flights?)
print("\n1. AIRCRAFT:")
print(suspicious['flightactualAircraftCabinName'].value_counts())

# 2. OPERATOR FRAUD CHECK
print("\n2. OPERATORS (any single bad actor?):")
print(suspicious['operatorName'].value_counts().head(10))

# 3. DISTANCE PROXY (same airport roundtrips?)
print("\n3. ZERO-DISTANCE (airport loops):")
print(suspicious['flightoriginAirportId'].value_counts().head(10))

# 4. FLIGHT HOURS (estimated vs actual mismatch)
print("\n4. ESTIMATED FLIGHT HOURS:")
print(suspicious['flightEstimatedFlightHours'].describe())

# 5. TAIL NUMBERS (same planes repeatedly?)
print("\n5. TOP TAIL NUMBERS:")
print(suspicious['flighttailNumber'].value_counts().head(10))

In [ ]:
# ------------------------------------------------------------
# EXTRACT Wheels Up Entities from operatorName (Fuzzy Match)
# ------------------------------------------------------------

print("🔍 Wheels Up Entities - UNIQUE MATCHES FOUND")
print("="*70)

# Your shortlist (case-insensitive)
wu_shortlist = [
    'Wheels Up Private Jets', 'Wheels Up LLC', 'Mountain Aviation', 
    'Alante Air Charter'
]

# Find ALL operatorNames containing these keywords
keywords = ['wheels up', 'mountain aviation', 'alante', 'gama aviation', 
           'tmc', 'twc aviation', 'wheels up aviation', 'wu']

wu_entities = []
for op in df_bookings['operatorName'].unique():
    op_lower = op.lower()
    if any(keyword in op_lower for keyword in keywords):
        count = len(df_bookings[df_bookings['operatorName'] == op])
        wu_entities.append((op, count))
        print(f"  '{op}' → {count:,} legs")

print(f"\n✅ FOUND {len(wu_entities)} Wheels Up entities")

# Sort by volume
wu_entities.sort(key=lambda x: x[1], reverse=True)

# Create clean list for Excel
print("\n📋 CLEAN LIST (copy to Excel):")
for name, count in wu_entities:
    print(f"{name}\t{count}")

In [ ]:
# ------------------------------------------------------------
# CONFIRM ALL "Wheels Up" / "WheelsUp" variants
# ------------------------------------------------------------

print("🔍 ALL operatorName containing 'wheels up' or 'wheelsup'")
print("="*70)

# Exact phrase match (case-insensitive)
wheels_up_mask = df_bookings['operatorName'].str.contains(
    'wheels up|wheelsup', 
    case=False, 
    na=False
)

wheels_up_operators = df_bookings[wheels_up_mask]['operatorName'].value_counts()

for op_name, count in wheels_up_operators.items():
    print(f"  '{op_name}' → {count:,} legs ({count/wheels_up_mask.sum()*100:.1f}%)")

print(f"\n✅ TOTAL Wheels Up legs: {wheels_up_mask.sum():,}")
print(f"✅ Wheels Up operators found: {len(wheels_up_operators)}")

# Create final filter list
wu_exact_names = wheels_up_operators.index.tolist()
print(f"\n📋 EXACT NAMES for filter: {wu_exact_names}")

In [ ]:
# ------------------------------------------------------------
# Check Flown flights MISSING origin/destination airports
# ------------------------------------------------------------

print("🚨 Flown flights MISSING airports")
print("="*60)

flown_data = df_bookings[df_bookings['flightStatus'] == 'Flown']

# Check origin airport missing
origin_missing = flown_data['flightoriginAirport'].isnull()
print(f"flightoriginAirport missing: {origin_missing.sum():,} ({origin_missing.mean()*100:.2f}%)")

# Check destination airport missing  
dest_missing = flown_data['flightdestinationAirport'].isnull()
print(f"flightdestinationAirport missing: {dest_missing.sum():,} ({dest_missing.mean()*100:.2f}%)")

# Both missing (IMMEDIATE RED FLAG)
both_missing = origin_missing & dest_missing
print(f"BOTH airports missing: {both_missing.sum():,} ({both_missing.mean()*100:.2f}%)")

# Show patterns if any missing
if origin_missing.sum() > 0:
    print("\n🔍 Origin missing patterns:")
    print(flown_data[origin_missing]['operatorName'].value_counts().head())

In [ ]:
# ------------------------------------------------------------
# Flown flights with NULL flightcost
# ------------------------------------------------------------

print("🚨 Flown flights with NULL flightcost")
print("="*50)

flown_null_cost = df_bookings[
    (df_bookings['flightStatus'] == 'Flown') & 
    (df_bookings['flightcost'].isnull())
]

null_count = len(flown_null_cost)
total_flown = len(df_bookings[df_bookings['flightStatus'] == 'Flown'])

print(f"Flown with NULL flightcost: {null_count:,}")
print(f"Percentage of Flown: {null_count/total_flown*100:.2f}%")
print(f"Total Flown legs: {total_flown:,}")

# Show patterns if any
if null_count > 0:
    print("\n🔍 Patterns in NULL cost Flown flights:")
    print("Top operators:")
    print(flown_null_cost['operatorName'].value_counts().head())
    print("\nTop aircraft:")
    print(flown_null_cost['flightactualAircraftCabinName'].value_counts().head())

In [ ]:
# ------------------------------------------------------------
# NULL COUNTS for Flown flights - CRITICAL FIELDS
# ------------------------------------------------------------

flown_data = df_bookings[df_bookings['flightStatus'] == 'Flown']

fields = [
    'flightId', 'atlasreservationid', 'flightStatus', 'legOrder',
    'flightoriginAirport', 'flightdestinationAirport', 
    'flightActualDepartureTime', 'flightActualArrivalTime', 
    'flightActualFlightHours', 'flightActualBilledHours',
    'flightactualAircraftCabinName', 'flightactualAircraftTypeName', 
    'operatorName', 'operatorType'
]

print("NULL % for Flown flights (SHOULD BE 0%)")
print("="*60)
print(f"{'Field':<30} | {'Null Count':>10} | {'Null %':>8}")
print("-"*60)

for field in fields:
    null_count = flown_data[field].isnull().sum()
    null_pct = (null_count / len(flown_data)) * 100
    print(f"{field:<30} | {null_count:>10,} | {null_pct:>7.2f}%")

print(f"\nTotal Flown legs: {len(flown_data):,}")

In [ ]:
# ------------------------------------------------------------
# Wheels Up Flown flights - NULL % for time fields
# ------------------------------------------------------------

# Filter Wheels Up + Flown
wu_flown = df_bookings[
    df_bookings['operatorName'].isin([
        'Wheels Up Private Jets, LLC', 
        'Wheels Up Partners, LLC'
    ]) & 
    (df_bookings['flightStatus'] == 'Flown')
]

fields = [
    'flightActualDepartureTime', 
    'flightActualArrivalTime', 
    'flightActualFlightHours', 
    'flightActualBilledHours'
]

print("Wheels Up Flown - NULL % (Time Fields)")
print("="*60)
print(f"{'Field':<28} | {'Null Count':>10} | {'Null %':>8}")
print("-"*60)

total_wu_flown = len(wu_flown)
for field in fields:
    null_count = wu_flown[field].isnull().sum()
    null_pct = (null_count / total_wu_flown) * 100
    print(f"{field:<28} | {null_count:>10,} | {null_pct:>7.2f}%")

print(f"\nTotal Wheels Up Flown legs: {total_wu_flown:,}")

In [ ]:
# ------------------------------------------------------------
# Wheels Up Flown - COMPLETE NULL ANALYSIS
# ------------------------------------------------------------

# Filter Wheels Up + Flown
wu_flown = df_bookings[
    df_bookings['operatorName'].isin([
        'Wheels Up Private Jets, LLC', 
        'Wheels Up Partners, LLC'
    ]) & 
    (df_bookings['flightStatus'] == 'Flown')
]

fields = [
    'flightActualDepartureTime', 'flightActualArrivalTime', 
    'flightActualFlightHours', 'flightActualBilledHours',
    'flightactualAircraftCabinName', 'flightactualAircraftTypeName',
    'flightId', 'atlasreservationid', 'flightStatus', 'legOrder',
    'flightoriginAirport', 'flightdestinationAirport', 
    'operatorName'
]

print("Wheels Up Flown - NULL % (ALL CRITICAL FIELDS)")
print("="*70)
print(f"{'Field':<30} | {'Null Count':>10} | {'Null %':>8}")
print("-"*70)

total_wu_flown = len(wu_flown)
for field in fields:
    null_count = wu_flown[field].isnull().sum()
    null_pct = (null_count / total_wu_flown) * 100
    print(f"{field:<30} | {null_count:>10,} | {null_pct:>7.2f}%")

print(f"\n✅ Total Wheels Up Flown legs: {total_wu_flown:,}")

In [ ]:
# ------------------------------------------------------------
# Missing times by OPERATOR (Flown flights only)
# ------------------------------------------------------------

flown_data = df_bookings[df_bookings['flightStatus'] == 'Flown']

time_fields = ['flightActualDepartureTime', 'flightActualArrivalTime']

print("🚨 MISSING TIMES BY OPERATOR (Flown flights)")
print("="*80)
print(f"{'Operator':<35} | {'Dep Null':>8} | {'Arr Null':>8} | {'Dep %':>7} | {'Arr %':>7}")
print("-"*80)

for op in flown_data['operatorName'].value_counts().head(20).index:
    op_flown = flown_data[flown_data['operatorName'] == op]
    total = len(op_flown)
    
    dep_null = op_flown['flightActualDepartureTime'].isnull().sum()
    arr_null = op_flown['flightActualArrivalTime'].isnull().sum()
    
    dep_pct = (dep_null / total) * 100
    arr_pct = (arr_null / total) * 100
    
    print(f"{op:<35} | {dep_null:>8,} | {arr_null:>8,} | {dep_pct:>6.1f}% | {arr_pct:>6.1f}%")

print(f"\nTotal Flown legs: {len(flown_data):,}")

In [ ]:
# ------------------------------------------------------------
# ALL FLIGHTSTATUS='Flown' + flightcost - NULL ANALYSIS
# ------------------------------------------------------------

flown_all = df_bookings[df_bookings['flightStatus'] == 'Flown']

fields = [
    'flightId', 'atlasreservationid', 'flightStatus', 'legOrder',
    'flightoriginAirport', 'flightdestinationAirport',
    'flightActualDepartureTime', 'flightActualArrivalTime',
    'flightActualFlightHours', 'flightActualBilledHours',
    'flightactualAircraftCabinName', 'flightactualAircraftTypeName',
    'operatorName', 'operatorType', 'flightcost'
]

print("Field\tNull Count\tNull %")
print("-"*50)

total_flown = len(flown_all)
for field in fields:
    null_count = flown_all[field].isnull().sum()
    null_pct = (null_count / total_flown) * 100 if total_flown > 0 else 0
    print(f"{field}\t{null_count:,}\t{null_pct:.2f}%")

print(f"\nTotal Flown legs (ALL): {total_flown:,}")

In [ ]:
# ------------------------------------------------------------
# WHEELS UP ONLY (2 operators) + flightcost - NULL ANALYSIS
# ------------------------------------------------------------

wu_flown = df_bookings[
    (df_bookings['operatorName'].isin([
        'Wheels Up Private Jets, LLC', 
        'Wheels Up Partners, LLC'
    ])) & 
    (df_bookings['flightStatus'] == 'Flown')
]

fields = [
    'flightId', 'atlasreservationid', 'flightStatus', 'legOrder',
    'flightoriginAirport', 'flightdestinationAirport',
    'flightActualDepartureTime', 'flightActualArrivalTime',
    'flightActualFlightHours', 'flightActualBilledHours',
    'flightactualAircraftCabinName', 'flightactualAircraftTypeName',
    'operatorName', 'operatorType', 'flightcost'
]

print("Field\tNull Count\tNull %")
print("-"*50)

total_wu = len(wu_flown)
for field in fields:
    null_count = wu_flown[field].isnull().sum()
    null_pct = (null_count / total_wu) * 100 if total_wu > 0 else 0
    print(f"{field}\t{null_count:,}\t{null_pct:.2f}%")

print(f"\nTotal Wheels Up Flown legs: {total_wu:,}")

In [ ]:
# 1. Print simple lists of all columns
print(f"{'='*20} df_bookings Columns ({len(df_bookings.columns)}) {'='*20}")
print(df_bookings.columns.tolist())

print(f"\n{'='*20} df_quotes Columns ({len(df_quotes.columns)}) {'='*20}")
print(df_quotes.columns.tolist())

# 2. Check for common columns (The "Bridge" for your Marry logic)
common_cols = set(df_bookings.columns).intersection(set(df_quotes.columns))
print(f"\n{'='*20} Potential Join Keys (Common Columns) {'='*20}")
print(common_cols if common_cols else "No identical column names found.")

In [13]:
# 1. Print simple lists of all columns
print(f"{'='*20} df_bookings Columns ({len(df_bookings.columns)}) {'='*20}")
print(df_bookings.columns.tolist())

print(f"\n{'='*20} df_quotes Columns ({len(df_quotes.columns)}) {'='*20}")
print(df_quotes.columns.tolist())

# 2. Check for common columns (The "Bridge" for your Marry logic)
common_cols = set(df_bookings.columns).intersection(set(df_quotes.columns))
print(f"\n{'='*20} Potential Join Keys (Common Columns) {'='*20}")
print(common_cols if common_cols else "No identical column names found.")

==================== df_bookings Columns (43) ====================
['flightId', 'atlasreservationid', 'flightStatus', 'legOrder', 'flightoriginAirportId', 'flightoriginAirport', 'flightoriginAirportName', 'flightoriginAirportCity', 'flightoriginAirportState', 'flightoriginAirportCountry', 'flightdestinationAirportId', 'flightdestinationAirport', 'flightdestinationAirportName', 'flightdestinationAirportCity', 'flightdestinationAirportState', 'flightdestinationAirportCountry', 'flightEstimatedDepartureTime', 'flightEstimatedArrivalTime', 'flightEstimatedFlightHours', 'flightEstimatedBilledHours', 'flightActualDepartureTime', 'flightActualArrivalTime', 'flightActualFlightHours', 'flightActualBilledHours', 'flightrequestedAircraftType', 'flightrequestedAircraftCabinId', 'flightrequestedAircraftCabinName', 'flightrequestedAircraftTypeId', 'flightrequestedAircraftTypeName', 'flighttailNumber', 'flightactualAircraftCabinId', 'flightactualAircraftCabinName', 'flightactualAircraftId', 'flightac

In [14]:
# 1. Check for exact Column Name overlaps (should be empty based on your list)
exact_matches = set(df_bookings.columns).intersection(set(df_quotes.columns))
print(f"Exact Column Name Matches: {exact_matches}")

# 2. THE SMOKING GUN: Check Value Overlap between your suspected keys
# We want to see how many IDs in Quotes actually exist in Bookings
quotes_keys = set(df_quotes['opportunityReservationId'].astype(str).unique())
bookings_keys = set(df_bookings['reservationId'].astype(str).unique())

intersection = quotes_keys.intersection(bookings_keys)

print(f"\n--- Join Key Analysis ---")
print(f"Unique Reservation IDs in Quotes:   {len(quotes_keys):,}")
print(f"Unique Reservation IDs in Bookings: {len(bookings_keys):,}")
print(f"Common IDs (The Match):             {len(intersection):,}")

# 3. Match Rate %
if len(quotes_keys) > 0:
    match_rate = (len(intersection) / len(quotes_keys)) * 100
    print(f"Match Rate: {match_rate:.2f}% of Quotes have a corresponding Booking record.")

# 4. Sample the overlap to be 100% sure the strings look the same
if len(intersection) > 0:
    print(f"\nSample of matching IDs: {list(intersection)[:5]}")

Exact Column Name Matches: set()

--- Join Key Analysis ---
Unique Reservation IDs in Quotes:   2,107
Unique Reservation IDs in Bookings: 52,845
Common IDs (The Match):             0
Match Rate: 0.00% of Quotes have a corresponding Booking record.


In [15]:
# 1. Peek at the raw values
print("--- Raw Value Inspection ---")
print(f"Bookings ID Sample: {df_bookings['reservationId'].dropna().unique()[:5]}")
print(f"Quotes ID Sample:   {df_quotes['opportunityReservationId'].dropna().unique()[:5]}")

# 2. Check Data Types (e.g., int64 vs object)
print(f"\n--- Data Type Check ---")
print(f"Bookings ID Type: {df_bookings['reservationId'].dtype}")
print(f"Quotes ID Type:   {df_quotes['opportunityReservationId'].dtype}")

# 3. Check for hidden Whitespace or 'NaN' strings
print(f"\n--- Formatting Check ---")
sample_quote_id = str(df_quotes['opportunityReservationId'].iloc[0])
print(f"Sample Quote ID Length: {len(sample_quote_id)} characters")

--- Raw Value Inspection ---
Bookings ID Sample: [794518 794521 794523 794528 794530]
Quotes ID Sample:   [845380. 845358. 845348. 845402. 845390.]

--- Data Type Check ---
Bookings ID Type: int64
Quotes ID Type:   float64

--- Formatting Check ---
Sample Quote ID Length: 3 characters


In [16]:
# 1. Clean and convert to string, removing the ".0" from floats
def clean_id(val):
    if pd.isna(val): return "NAN"
    # Convert to float first, then int, then string to strip ".0"
    return str(int(float(val)))

df_bookings['res_id_clean'] = df_bookings['reservationId'].apply(clean_id)
df_quotes['res_id_clean'] = df_quotes['opportunityReservationId'].apply(clean_id)

# 2. Check the Ranges
print("--- ID Range Overlap ---")
print(f"Bookings Range: {df_bookings['res_id_clean'].min()} to {df_bookings['res_id_clean'].max()}")
print(f"Quotes Range:   {df_quotes['res_id_clean'].min()} to {df_quotes['res_id_clean'].max()}")

# 3. Check for intersection again
bookings_set = set(df_bookings[df_bookings['res_id_clean'] != "NAN"]['res_id_clean'])
quotes_set = set(df_quotes[df_quotes['res_id_clean'] != "NAN"]['res_id_clean'])
overlap = bookings_set.intersection(quotes_set)

print(f"\nNew Match Count: {len(overlap):,}")

--- ID Range Overlap ---
Bookings Range: 794518 to 847634
Quotes Range:   825958 to NAN

New Match Count: 2,100


In [17]:
# Count total quotes vs flown bookings
total_quotes = len(df_quotes)
total_flown = len(df_bookings[df_bookings['flightStatus'] == 'Flown'])

print(f"Total Rows in Quotes: {total_quotes:,}")
print(f"Total Flown Flights:  {total_flown:,}")

# Check the ratio
if total_quotes < total_flown:
    print(f"\nWarning: You have quotes for only { (total_quotes/total_flown)*100 :.2f}% of your flown data.")

Total Rows in Quotes: 31,077
Total Flown Flights:  220,957



In [18]:
# Check the status of all 31,077 quotes
print("--- Quote Option Status Distribution ---")
quote_status_counts = df_quotes['quoteOptionStatus'].value_counts()
quote_status_pct = (df_quotes['quoteOptionStatus'].value_counts(normalize=True) * 100)

funnel_report = pd.DataFrame({
    'Count': quote_status_counts,
    'Percentage (%)': quote_status_pct
})
print(funnel_report)

--- Quote Option Status Distribution ---
                   Count  Percentage (%)
quoteOptionStatus                       
ACCEPTED            4791       78.890170
EXPIRED              683       11.246501
REVOKED              558        9.188210
READY                 28        0.461057
DECLINED              13        0.214062


In [19]:
# 1. Convert quote dates to datetime
df_quotes['opportunityCreateDate'] = pd.to_datetime(df_quotes['opportunityCreateDate'], errors='coerce')

# 2. Get the Date Range for Quotes
quote_start = df_quotes['opportunityCreateDate'].min()
quote_end = df_quotes['opportunityCreateDate'].max()

print(f"--- Quote Timeline ---")
print(f"Quotes start on: {quote_start}")
print(f"Quotes end on:   {quote_end}")

# 3. See how many Bookings exist in this SAME time window
# (Using reservationCreateDate since it aligns with the quote phase)
df_bookings['reservationCreateDate'] = pd.to_datetime(df_bookings['reservationCreateDate'], errors='coerce')

bookings_in_window = df_bookings[
    (df_bookings['reservationCreateDate'] >= quote_start) & 
    (df_bookings['reservationCreateDate'] <= quote_end)
]

print(f"\n--- Comparison in this Window ---")
print(f"Bookings created in this period: {len(bookings_in_window):,}")
print(f"Quotes created in this period:   {len(df_quotes):,}")

if len(bookings_in_window) > 0:
    coverage = (len(df_quotes) / len(bookings_in_window)) * 100
    print(f"Effective Coverage in Window:    {coverage:.2f}%")

--- Quote Timeline ---
Quotes start on: 2024-04-01 01:43:54+00:00
Quotes end on:   2026-04-01 22:57:56+00:00

--- Comparison in this Window ---
Bookings created in this period: 258,370
Quotes created in this period:   31,077
Effective Coverage in Window:    12.03%


In [20]:
# 1. Flag the flights that have a matching quote
df_bookings['has_quote'] = df_bookings['res_id_clean'].isin(quotes_set)

# 2. Compare average costs and standard deviation
price_comparison = df_bookings[df_bookings['flightStatus'] == 'Flown'].groupby('has_quote')['flightcost'].agg(['mean', 'std', 'count'])

print("--- Price Behavior: Marketplace (With Quote) vs. Core (No Quote) ---")
print(price_comparison)

--- Price Behavior: Marketplace (With Quote) vs. Core (No Quote) ---
                   mean           std   count
has_quote                                    
False      15076.068485   9210.896616  212384
True       21933.401856  14313.067662    8064
